# Lab 2: Wrapping the Assistant in an API, and Calling It from an In-House App

**Time:** about 60 minutes

A notebook is where you build an assistant. It isn't where people use it. For the member-service desk, the call-centre
system or the staff intranet to use the handbook assistant, it has to be a **service**: running all the time, reachable over
the network, and answering requests from other systems in a fixed format.

### What is an API, and why does the assistant need one?

An **API** (application programming interface) is a way for one program to call another. You've used one all week:
every time your notebook sent a request to Ollama at `localhost:11434/api/chat`, it was calling Ollama's API. Today you'll
build the same kind of thing for the handbook assistant, so that any Fund system can send it a question and get an
answer back, without knowing anything about FAISS, embeddings or prompts.

Think of it like a service counter: other departments don't need to know how the back office works. They hand over a
request in an agreed format, and get a reply in an agreed format.

You'll build that service with **FastAPI**, a popular Python library for building APIs, and add the three things any
internal API at the Fund needs before it goes live:

- **authentication:** only known applications can call it (API keys);
- **rate limiting:** one misbehaving caller can't swamp the service, or the LLM (large language model) behind it;
- **logging:** every request is recorded, without copying members' questions into yet another place.

Then you'll call it from a small "in-house application", the way a real system would.

```
member-service desk (browser page / desk_app.py)
        │  HTTPS, X-API-Key header
        ▼
FastAPI service ── auth ── rate limit ── guardrail ── retrieval (FAISS) ── LLM (Ollama) ── log
```

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **API** | a way for one program to call another, with an agreed request and reply format |
| **service / server** | a program that runs all the time, waiting for requests |
| **endpoint** | one address on the service, e.g. `/v1/ask`, with its own job |
| **HTTP** | the protocol web browsers and APIs use; HTTPS is the encrypted version |
| **GET / POST** | the two common kinds of request: GET asks for something, POST sends data |
| **header** | a labelled extra piece of information sent with a request, e.g. `X-API-Key` |
| **API key** | a secret string that identifies which application is calling |
| **status code** | the three-digit number at the start of every reply: 200 = OK, 401 = not allowed ... |
| **rate limit** | the most requests a caller may send per minute |
| **hash** | a fixed-length fingerprint of some text; the same text always gives the same hash, but you can't get the text back |
| **FastAPI** | a Python library for building APIs |
| **uvicorn** | the web server program that runs a FastAPI app |
| **Pydantic** | a library that checks the shape of requests and replies |
| **environment variable** | a setting given to a program from outside its code, e.g. the API keys |
| **class** | a bundle of data and the functions that work on it |

## Where this lab fits

**What you've done so far.** In Lab 1 you built a handbook assistant with RAG: guardrail, hybrid retrieval of current
passages, a relevance threshold, answers with citations, and an end-to-end test. You saved the index, the chunks and every
setting to `rag_index/`. All week you've been a client of another service's API: Ollama at `localhost:11434`.

**What you'll do now.** You'll turn the assistant into a service that other Fund systems can call. You'll move the Lab 1
code into a module, wrap it in a FastAPI app with API keys, a rate limit and privacy-aware logging, test every safeguard
without a server, run it as a real server, call it from a small desk application, measure its latency, and list what
would change before it went live.

**Why it matters.** A notebook helps the person who wrote it. A service helps everyone: the member-service desk, the call
centre and the staff intranet can all ask the same assistant, in the same way, with the same safeguards and the same
record of what was asked.

**What happens next.** Lab 3 compares running this service on the Fund's own servers with running it on a cloud platform.
On Day 10 you'll deploy a service like this in a container, watch it with monitoring dashboards, and govern it.

### Theory: from a notebook to a service

| | Notebook | Service |
|---|---|---|
| Who uses it | the person running the cells | any program that can send an HTTP request |
| When it runs | when someone clicks | all the time, waiting for requests |
| Loading models | every time the notebook starts | once, at start-up; then many answers |
| Input | whatever you type | checked against a fixed format before any code runs |
| Who may call it | anyone with the notebook | only applications with a valid key |
| Record of use | none | a log line per request |

The design principle is **separation**: the service hides how it works (FAISS, embeddings, prompts) behind a small,
stable **contract**: send a question in this shape, get an answer, sources and a request id back in that shape. The team
running the call-centre system needs to know the contract, not the internals. That's also why the address has a version,
`/v1/`: the contract can change later as `/v2/` without breaking anyone who relies on `/v1/`.

## Imports and checks

This lab needs Lab 1's saved index (`rag_index/`), so it stops with a clear message if that isn't there. The service's
code will be written into its own folder, `rag_service`, the way it would be deployed.

New tools: `subprocess` (to start the server and run the desk app as separate programs) and `sys` (to find the current
Python and tell it where the service code is).

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import requests

if not Path("rag_index/chunks.faiss").exists():
    raise FileNotFoundError("Run Lab 1 first: it saves the index to rag_index/")

# The service's code will live in its own folder.
Path("rag_service").mkdir(exist_ok=True)

for path in sorted(Path("rag_index").iterdir()):
    print("index file:", path.name)

> **Where we are.** Lab 1's index is found, and the `rag_service` folder is ready for the service's code. First, move the
> assistant out of the notebook into a file the server can import.

## 1. The assistant as a module

Everything from Lab 1 that answers a question goes into one **class**, loaded once when the service starts. Loading the
embedding model and the index takes seconds. Answering a question should take well under that, so you load once and
answer many times.

A class bundles data and the functions that use it. `__init__` runs once, when the object is created, and stores
everything on `self`. The other functions (methods) can then use `self.chunks`, `self.index` and so on. The method
bodies are the Lab 1 code, almost line for line:

| Method | What it does | Lab 1 equivalent |
|---|---|---|
| `__init__` | loads the settings, chunks, index, embedding model and TF-IDF | Sections 1-4, 9 |
| `dense_search` | the k chunks closest in meaning | `dense_search` |
| `keyword_search` | the k chunks sharing the most distinctive words | `keyword_search` |
| `search` | hybrid search, with superseded passages removed | `hybrid_search` + `current_search` |
| `llm_available` | is Ollama answering? (for the health check) | (new) |
| `answer` | guardrail, retrieval, threshold, LLM, citations | `assistant` + `rag_answer` |

One small difference: the searches work with chunk **positions** (numbers) rather than the chunks themselves, which is
slightly simpler.

The first line, `%%writefile rag_service/rag_core.py`, is a Jupyter instruction: instead of running the cell, it
**saves** the cell's contents as a Python file. That file is what the server will use.

In [ ]:
%%writefile rag_service/rag_core.py
"""The handbook assistant: guardrail -> hybrid retrieval (current documents only) -> grounded answer with citations."""
import json
import os
import re
from pathlib import Path

import faiss
import numpy as np
import requests
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer


class RagAssistant:
    """The Lab 1 assistant as one object: load everything once, then answer many questions."""

    def __init__(self, index_dir="rag_index"):
        folder = Path(index_dir)

        # The saved settings, chunks and vectors from Lab 1.
        self.cfg = json.loads((folder / "config.json").read_text(encoding="utf-8"))
        self.chunks = json.loads((folder / "chunks.json").read_text(encoding="utf-8"))
        self.index = faiss.read_index(str(folder / "chunks.faiss"))

        # The embedding model, and a TF-IDF model fitted on the chunk texts (for keyword search).
        self.embedder = SentenceTransformer(self.cfg["embed_model"])
        chunk_texts = []
        for chunk in self.chunks:
            chunk_texts.append(chunk["text"])
        self.tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
        self.matrix = self.tfidf.fit_transform(chunk_texts)

        # The personal-data guardrail pattern, and where Ollama is.
        self.personal = re.compile(self.cfg["personal_pattern"], re.IGNORECASE)
        self.ollama_url = os.environ.get("OLLAMA_URL", "http://localhost:11434")

    def dense_search(self, question, k):
        """[(chunk position, similarity), ...] for the k chunks closest in meaning."""
        vector = self.embedder.encode(["query: " + question], normalize_embeddings=True).astype(np.float32)
        scores, positions = self.index.search(vector, k)
        results = []
        for position, score in zip(positions[0], scores[0]):
            results.append((int(position), float(score)))
        return results

    def keyword_search(self, question, k):
        """[(chunk position, similarity), ...] for the k chunks sharing the most distinctive words."""
        similarities = (self.matrix @ self.tfidf.transform([question]).T).toarray().ravel()
        results = []
        for position in np.argsort(-similarities)[:k]:
            results.append((int(position), float(similarities[position])))
        return results

    def search(self, question, k=4, pool=20):
        """Hybrid search (reciprocal rank fusion), superseded passages removed. Returns [(chunk, dense similarity), ...]."""
        dense = self.dense_search(question, pool)
        keyword = self.keyword_search(question, pool)

        # Reciprocal rank fusion: 1 / (60 + rank) points from each list.
        points = {}
        for results in [dense, keyword]:
            for rank, (position, score) in enumerate(results):
                points[position] = points.get(position, 0.0) + 1 / (60 + rank)

        dense_similarity = {}
        for position, score in dense:
            dense_similarity[position] = score

        # Best first, skipping superseded passages.
        merged = []
        for position in sorted(points, key=points.get, reverse=True):
            chunk = self.chunks[position]
            if not chunk.get("superseded_by"):
                merged.append((chunk, dense_similarity.get(position, 0.0)))
        return merged[:k]

    def llm_available(self):
        """True if Ollama answers."""
        try:
            return requests.get(self.ollama_url + "/api/version", timeout=3).ok
        except requests.exceptions.RequestException:
            return False

    def answer(self, question):
        """Guardrail, then retrieval, then a grounded answer with citations (Lab 1, Sections 6-7)."""
        # 1. Personal-record questions never reach the LLM.
        if self.personal.search(question):
            return {"answer": self.cfg["personal_reply"], "sources": [], "route": "personal-data guardrail"}

        # 2. Retrieve; refuse if nothing is relevant enough.
        results = self.search(question, k=self.cfg["k"])
        best_score = 0.0
        for chunk, score in results:
            best_score = max(best_score, score)
        if len(results) == 0 or best_score < self.cfg["min_score"]:
            return {"answer": self.cfg["refusal"], "sources": [], "route": "below relevance threshold"}

        # 3. Numbered sources, then ask the LLM.
        context = ""
        for number, (chunk, score) in enumerate(results, start=1):
            context = (context + "[" + str(number) + "] (" + chunk["source"] + ", " + chunk["section"] + ", dated "
                       + chunk["date"] + ")\n" + chunk["text"] + "\n\n")
        body = {
            "model": self.cfg["llm"],
            "stream": False,
            "options": {"temperature": 0, "num_predict": 250, "seed": 42},
            "messages": [
                {"role": "system", "content": self.cfg["system_prompt"]},
                {"role": "user", "content": "Sources:\n" + context + "Question: " + question},
            ],
        }
        response = requests.post(self.ollama_url + "/api/chat", json=body, timeout=120)
        response.raise_for_status()          # stop with an error if Ollama returned one
        text = response.json()["message"]["content"].strip()

        # 4. The sources it cited.
        cited = set()
        for found in re.findall(r"\[(\d+)\]", text):
            if 1 <= int(found) <= len(results):
                cited.add(int(found))
        sources = []
        for n in sorted(cited):
            chunk = results[n - 1][0]
            sources.append(chunk["source"] + " " + chunk["section"])

        return {"answer": text, "sources": sources, "route": "answered"}

> **Where we are.** `rag_core.py` holds the whole assistant in one class: everything is loaded once in `__init__`, and
> `answer` runs guardrail, retrieval, threshold, LLM and citations. Now put a web service in front of it.

### Theory: the three safeguards every internal API needs

- **Authentication** answers *"who is calling?"*. An API key is a long secret string given to each calling application.
  The service rejects requests without a known key (`401`). It also tells you which application sent each request, which
  matters for the log and for rate limits.
- **Rate limiting** answers *"is this caller asking too much?"*. The LLM behind the service can only answer a few
  questions at a time. Without a limit, one buggy client stuck in a loop could make the service unusable for everyone.
  Each client gets its own allowance per minute; over it, they get `429` and are told when to retry.
- **Logging** answers *"what happened?"*: who called, when, how long it took, which route the question took. It's how you
  monitor the service and investigate complaints. But logs are copied and kept for years, so they must not become a
  second, uncontrolled store of members' personal data. Hashing the question keeps the useful part (counting repeats)
  without the risky part (the text).

## 2. The service

Read the file below before running it. Each of the three safeguards is only a few lines:

- `require_key`: every request must carry an `X-API-Key` header that matches a known client. The keys come from an
  environment variable, never from the code, so they can be different in test and production and never end up in Git.
- `check_rate_limit`: remembers when each client called. More than `RATE_LIMIT_PER_MINUTE` requests in a minute gets
  `429 Too Many Requests` and a `Retry-After` header, which well-behaved clients respect.
- `log_request`: one JSON (JavaScript Object Notation, a plain-text format for structured data) line per request with
  who called, how long it took, which route it took and how many sources it cited. It stores a **hash** of the
  question, not the question. Members sometimes type personal details into questions, and logs are copied, shipped and
  kept for years. The hash still lets you count repeated questions.

Two FastAPI ideas you'll see:

- `@app.get("/health")` and `@app.post("/v1/ask")` say "this function answers requests to this address". The `@` line
  above a function is called a **decorator**: it registers the function with the app. `/v1/` is a version number in the
  address, so a future `/v2/ask` can change the format without breaking existing callers.
- `BaseModel` classes (from Pydantic) describe the shape of the request and the reply. FastAPI checks every request
  against them before your code runs. `Field(..., min_length=3, max_length=500)` means the question is required and must
  be 3 to 500 characters.

`Depends(require_key)` in the `ask` function tells FastAPI: "before running this, run `require_key`, and give me what it
returns" (the caller's name). The endpoints are:

| Endpoint | Purpose |
|---|---|
| `GET /health` | is the service up, how many chunks it has, can it reach the LLM |
| `POST /v1/ask` | the main one: a question in, an answer with sources out |
| `GET /desk` | a minimal web page for the member-service desk |

Again, `%%writefile` saves the cell as a file rather than running it.

In [ ]:
%%writefile rag_service/main.py
"""Internal handbook-assistant API.

Run:  uvicorn main:app --app-dir rag_service --port 8000
Env:  API_KEYS="desk:<key>,callcentre:<key>"   RATE_LIMIT_PER_MINUTE=20   OLLAMA_URL=http://localhost:11434
"""
import hashlib
import json
import os
import time
import uuid
from pathlib import Path

from fastapi import Depends, FastAPI, Header, HTTPException
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field

from rag_core import RagAssistant

# ---- Settings, from environment variables ----

# API_KEYS="desk:abc123,callcentre:xyz789"  ->  {"abc123": "desk", "xyz789": "callcentre"}
API_KEYS = {}
for item in os.environ.get("API_KEYS", "").split(","):
    if ":" in item:
        client_name, key = item.split(":", 1)
        API_KEYS[key] = client_name

RATE_LIMIT = int(os.environ.get("RATE_LIMIT_PER_MINUTE", "20"))
LOG_FILE = Path(os.environ.get("REQUEST_LOG", "logs/requests.jsonl"))
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

# ---- The app, and the assistant (loaded once, when the service starts) ----

app = FastAPI(title="Handbook Assistant API", version="1.0",
              description="Answers staff questions from the Member Handbook (Training Edition) and circulars.")
assistant = RagAssistant(index_dir=os.environ.get("RAG_INDEX_DIR", "rag_index"))


# ---- The shape of a request and of a reply ----

class Question(BaseModel):
    question: str = Field(..., min_length=3, max_length=500, examples=["How much is the funeral grant?"])


class Answer(BaseModel):
    answer: str
    sources: list[str]
    route: str
    request_id: str
    latency_ms: int


# ---- Safeguard 1: API keys ----

def require_key(x_api_key: str | None = Header(default=None)):
    """Return the caller's name if its X-API-Key header is known; otherwise refuse with 401."""
    if x_api_key not in API_KEYS:
        raise HTTPException(status_code=401, detail="Missing or invalid API key")
    return API_KEYS[x_api_key]


# ---- Safeguard 2: rate limiting ----

recent_calls = {}      # client name -> times of its calls in the last 60 seconds


def check_rate_limit(client):
    """Refuse with 429 if this client has already made RATE_LIMIT calls in the last 60 seconds."""
    now = time.time()

    # Keep only the calls from the last 60 seconds.
    calls = []
    for call_time in recent_calls.get(client, []):
        if now - call_time <= 60:
            calls.append(call_time)

    if len(calls) >= RATE_LIMIT:
        wait = int(60 - (now - calls[0])) + 1                 # seconds until the oldest call is a minute old
        raise HTTPException(status_code=429, detail="Rate limit exceeded", headers={"Retry-After": str(wait)})

    calls.append(now)
    recent_calls[client] = calls


# ---- Safeguard 3: logging ----

def log_request(entry):
    """Append one line to the log: who, when, how long, which route. Never the question text."""
    entry["ts"] = time.strftime("%Y-%m-%dT%H:%M:%S")
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")


# ---- The endpoints ----

@app.get("/health")
def health():
    return {
        "status": "ok",
        "chunks_indexed": len(assistant.chunks),
        "llm_reachable": assistant.llm_available(),
        "llm": assistant.cfg["llm"],
        "index_built": assistant.cfg.get("built"),
    }


@app.post("/v1/ask", response_model=Answer)
def ask(q: Question, client: str = Depends(require_key)):
    """The main endpoint. Depends(require_key) runs the key check first and passes in the client's name."""
    check_rate_limit(client)
    request_id = uuid.uuid4().hex[:12]            # a short random id, to find this request in the log
    start = time.perf_counter()

    try:
        result = assistant.answer(q.question)
    except Exception as error:
        # Never send internal error details to the caller; log the type of error instead.
        log_request({"request_id": request_id, "client": client, "route": "error", "error": type(error).__name__})
        raise HTTPException(status_code=503, detail="The assistant is temporarily unavailable")

    latency = int((time.perf_counter() - start) * 1000)

    # A fingerprint of the question, not the text: the same question always gives the same hash.
    question_hash = hashlib.sha256(q.question.strip().lower().encode()).hexdigest()[:16]
    log_request({
        "request_id": request_id,
        "client": client,
        "route": result["route"],
        "latency_ms": latency,
        "n_sources": len(result["sources"]),
        "question_chars": len(q.question),
        "question_hash": question_hash,
    })

    return Answer(answer=result["answer"], sources=result["sources"], route=result["route"],
                  request_id=request_id, latency_ms=latency)


# ---- A minimal web page for the member-service desk ----

DESK_PAGE = """<!doctype html><html><head><meta charset="utf-8"><title>Member Service Desk</title>
<style>body{font-family:system-ui,sans-serif;max-width:720px;margin:2rem auto;padding:0 1rem}
input,textarea,button{font:inherit;width:100%;margin:.3rem 0;padding:.5rem}#out{white-space:pre-wrap;background:#f4f4f2;padding:1rem}
</style></head><body><h2>Member Service Desk: handbook assistant</h2>
<label>API key <input id="key" type="password"></label>
<label>Question <textarea id="q" rows="3">Msaada wa mazishi ni kiasi gani?</textarea></label>
<button onclick="ask()">Ask</button><div id="out"></div>
<script>
async function ask(){const r=await fetch('/v1/ask',{method:'POST',headers:{'Content-Type':'application/json',
'X-API-Key':document.getElementById('key').value},body:JSON.stringify({question:document.getElementById('q').value})});
const d=await r.json();document.getElementById('out').textContent=r.ok?d.answer+'\\n\\nSources: '+(d.sources.join('; ')||'none')+
'\\n('+d.latency_ms+' ms, '+d.route+')':'Error '+r.status+': '+d.detail;}
</script></body></html>"""


@app.get("/desk", response_class=HTMLResponse)
def desk():
    return DESK_PAGE

> **Where we are.** `main.py` defines the app: three endpoints, request and reply formats checked by Pydantic, an API-key
> check, a per-client rate limit and a log line per request. Before running it as a server, test it directly.

## 3. Test it without starting a server

FastAPI's `TestClient` calls the app directly, in the same process. It's the fastest way to test, and exactly what an
automated test suite would do (Day 10). The environment variables are set *before* the app is imported, the same way
they'd be set on a server. The rate limit is deliberately tiny here so you can see it work.

`sys.path.insert(0, ...)` adds the `rag_service` folder to the places Python looks for modules, so `import main` finds
the file you just wrote. Importing `main` runs it, which loads the assistant (a few seconds).

In [ ]:
# Settings for the test, as a server would get them.
os.environ["API_KEYS"] = "desk:desk-test-key-123,callcentre:cc-test-key-456"
os.environ["RATE_LIMIT_PER_MINUTE"] = "5"
os.environ["REQUEST_LOG"] = "logs/test_requests.jsonl"

# Let Python find main.py and rag_core.py in the rag_service folder.
sys.path.insert(0, str(Path("rag_service").resolve()))

from fastapi.testclient import TestClient
import main

client = TestClient(main.app)
print("health:", client.get("/health").json())

### The status codes you'll see

Every HTTP response starts with a three-digit status code. These are the ones that matter for this service:

| Code | Meaning | When this service sends it |
|---|---|---|
| 200 | OK | The question was answered |
| 401 | Unauthorized | No API key, or a wrong one |
| 422 | Unprocessable entity | The request broke the input rules (e.g. a question shorter than 3 characters) |
| 429 | Too many requests | This client went over its rate limit; wait and retry |
| 503 | Service unavailable | Something behind the API (e.g. the LLM) failed |

Try four requests: no key, a wrong key, a question that's too short, and a good one. The key goes in the request's
**headers**, a dictionary of extra information sent along with it.

In [ ]:
DESK_KEY = {"X-API-Key": "desk-test-key-123"}
question = {"question": "How much is the funeral grant?"}

r = client.post("/v1/ask", json=question)
print("no key:            ", r.status_code, r.json())

r = client.post("/v1/ask", json=question, headers={"X-API-Key": "guess"})
print("wrong key:         ", r.status_code, r.json())

r = client.post("/v1/ask", json={"question": "?"}, headers=DESK_KEY)
print("question too short:", r.status_code)

good = client.post("/v1/ask", json=question, headers=DESK_KEY)
print("good request:      ", good.status_code)
good.json()

`401` for a missing or wrong key, `422` for input that breaks the rules (Pydantic validates it before your code even runs),
and `200` with an answer, its sources, a request id and the time taken.

Now the rate limit. The desk client has used one of its five requests this minute (the request rejected with `422` never
reached the rate limiter). Send five more: four should succeed and the fifth be refused. Then show that the call-centre
client, with its own key, has its own separate allowance, and read the `Retry-After` header that tells the desk how long
to wait.

In [ ]:
# Five more requests from the desk: the first four fit in its limit of 5, the fifth doesn't.
for attempt in range(5):
    r = client.post("/v1/ask", json={"question": "What is the retirement age?"}, headers=DESK_KEY)
    print("desk request", attempt + 1, "->", r.status_code)

# The call centre has its own allowance.
r = client.post("/v1/ask", json={"question": "What is the retirement age?"}, headers={"X-API-Key": "cc-test-key-456"})
print("call-centre client:", r.status_code, "(its own limit, unaffected)")

# How long the desk must wait.
blocked = client.post("/v1/ask", json={"question": "x" * 10}, headers=DESK_KEY)
print("Retry-After:", blocked.headers.get("retry-after"), "seconds")

After its limit, the desk client gets `429`, while the call-centre client, with its own allowance, carries on.

Look at what was logged (the last four lines of the log file):

In [ ]:
log_lines = Path("logs/test_requests.jsonl").read_text().splitlines()
for line in log_lines[-4:]:
    print(line)

No question text in the log. Just enough to monitor the service (latency, routes, errors, which client) and to spot
repeated questions via the hash.

**Your turn (1).** Add an endpoint `GET /v1/sources/{source}` that returns the list of chunk sections indexed for a given
source file (for example `07_funeral_grant.md`), protected by the same API key. Test it with the `TestClient`. Why might
the desk team want this?

In [ ]:
# Your code here

> **Where we are.** Every safeguard works under test: wrong or missing keys get `401`, bad input gets `422`, the sixth
> request in a minute gets `429`, clients have separate allowances, and the log holds no question text. Now run it the
> way it would really run.

## 4. Run it as a real server

Now start the service properly with **uvicorn**, the web server FastAPI runs on. In a terminal you'd type:

```bash
# Windows PowerShell
$env:API_KEYS = "desk:choose-a-long-random-key"
uvicorn main:app --app-dir rag_service --port 8000

# macOS / Linux
API_KEYS="desk:choose-a-long-random-key" uvicorn main:app --app-dir rag_service --port 8000
```

`main:app` means "the object called `app` in the file `main.py`"; `--port` is the numbered door the server listens on.

The next cell does the same from the notebook, in the background, so you can keep working here.
`subprocess.Popen` starts a program and returns straight away, leaving it running. The server gets its own copy of the
environment variables, with a different key and a more generous rate limit. Its output is thrown away (`DEVNULL`), to
keep the notebook tidy.

In [ ]:
# The server's settings: a copy of this notebook's environment, plus our own values.
server_env = os.environ.copy()
server_env["API_KEYS"] = "desk:lab-key-2026"
server_env["RATE_LIMIT_PER_MINUTE"] = "30"
server_env["REQUEST_LOG"] = "logs/requests.jsonl"

# Popen starts the server and returns straight away, leaving it running in the background.
command = [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", "rag_service", "--port", "8010"]
server = subprocess.Popen(command, env=server_env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
BASE = "http://127.0.0.1:8010"

The server needs a few seconds to load the embedding model before it can answer. This loop asks `/health` once a
second, up to 120 times, and stops (`break`) as soon as the server replies. Until then each request fails with a
connection error, which the `except` part catches before waiting a second and trying again.

In [ ]:
# Loading the embedding model takes a few seconds. Ask /health once a second until the server answers.
for attempt in range(120):
    try:
        if requests.get(BASE + "/health", timeout=10).ok:
            break
    except requests.exceptions.RequestException:
        time.sleep(1)                       # not listening yet

print(requests.get(BASE + "/health").json())

While it's running, open **http://127.0.0.1:8010/docs** in your browser. FastAPI generates interactive documentation for
every endpoint, which is what the teams calling your service will read. And open **http://127.0.0.1:8010/desk**: that's a
minimal member-service-desk page calling the API. Enter the key `lab-key-2026` and ask a question.

> **Where we are.** The service is running on port 8010, its `/health` endpoint answers, and you can see its documentation
> and the desk page in a browser. Now call it the way another Fund system would.

## 5. The in-house application

Other Fund systems won't use a browser page. They'll call the API from their own code. Here's a small command-line "desk
application" of the kind the member-service team might script. Its `ask` function:

1. sends the question with the API key;
2. if the reply is `429`, reads `Retry-After`, waits that long, and tries again (up to 3 times);
3. for any other error, stops with a clear error (`raise_for_status`);
4. otherwise returns the reply.

`if __name__ == "__main__":` means "only run this part when the file is started as a program" (not when another file
imports it). `sys.argv[1:]` is the list of words typed after the program's name on the command line.

In [ ]:
%%writefile desk_app.py
"""Member-service desk helper: ask the handbook assistant from the command line.

Usage:  python desk_app.py "How much is the funeral grant?"
Env:    ASSISTANT_URL (default http://127.0.0.1:8000), ASSISTANT_KEY
"""
import os
import sys
import time

import requests

URL = os.environ.get("ASSISTANT_URL", "http://127.0.0.1:8000")
KEY = os.environ.get("ASSISTANT_KEY", "")


def ask(question, retries=3):
    """Call the assistant's API, waiting and retrying when told to slow down (429)."""
    for attempt in range(retries):
        response = requests.post(URL + "/v1/ask", json={"question": question}, headers={"X-API-Key": KEY}, timeout=60)

        if response.status_code == 429:
            # Respect the rate limit: wait as long as the server asks, then try again.
            wait_seconds = int(response.headers.get("Retry-After", "5"))
            time.sleep(wait_seconds)
            continue

        response.raise_for_status()          # stop with an error for any other failure
        return response.json()

    raise RuntimeError("Assistant busy; try again shortly")


if __name__ == "__main__":
    question = " ".join(sys.argv[1:])
    if question == "":
        question = "How much is the funeral grant?"

    reply = ask(question)
    print(reply["answer"])
    print("Sources:", "; ".join(reply["sources"]) or "none", "(" + str(reply["latency_ms"]) + " ms)")

Run the desk app as a separate program, three times, the way a member-service officer would from a terminal. The
server's address and key are passed as environment variables. `subprocess.run` waits for the program to finish, and
`capture_output=True` collects what it printed.

In [ ]:
# Run desk_app.py three times, as a separate program, pointed at our server.
desk_env = os.environ.copy()
desk_env["ASSISTANT_URL"] = BASE
desk_env["ASSISTANT_KEY"] = "lab-key-2026"

for question in ["Fao la uzazi linalipwa kwa muda gani?", "What is my contribution balance?",
                 "Can a pensioner verify on the app?"]:
    result = subprocess.run([sys.executable, "desk_app.py", question], env=desk_env,
                            capture_output=True, text=True, timeout=180)
    print("Q:", question)
    print(result.stdout or result.stderr)       # what it printed (or its error message)

Look closely at the Swahili question. On the test run it was **refused** ("I can't find this in the Fund's documents"),
even though the maternity section answers it and the same question in English is answered correctly. Why? The relevance
threshold in `rag_index/config.json` was set in Lab 1 using English test questions only. A Swahili question matched to an
English passage scores a little lower, falls under the bar, and never reaches the LLM.

That's a real finding, not a bug in the lab, and it's exactly the kind of problem that stays invisible if you only test in
English: the service works well for staff who ask in English and quietly fails those who ask in Swahili. The fixes, from
quickest to best: lower the threshold a little and re-check the unanswerable questions; set a separate threshold for
Swahili questions; or calibrate on a test set with plenty of Swahili questions in it from the start. Whichever you choose,
report accuracy **by language** from now on.

**Your turn (2).** Write five Swahili versions of questions from `rag_eval_questions.csv`, score them with the assistant's
`search` method (`main.assistant.search(q, k=1)`), and compare their best scores with the English originals. What threshold
would let them through without letting "What is the capital of Kenya?" through?

In [ ]:
# Your code here

Notice also that the client **handles 429 properly**: it waits as long as the server asks, then retries. Rate limiting only
works if clients cooperate like this, which is worth writing into the Fund's integration standards.

### How fast is it?

**Latency** is how long a caller waits for an answer. Time five questions through the real server, then look at the
**median** (the middle value, the typical wait) and the **maximum** (the worst wait).

### Theory: latency, throughput and capacity

Two numbers describe how fast a service is:

- **Latency**: how long one caller waits for one answer (here, a few seconds, mostly the LLM writing).
- **Throughput**: how many answers the service can give per minute.

If the LLM answers one question at a time, throughput is simply 60 seconds divided by the latency: at 3 seconds per answer,
about 20 answers a minute. Requests that arrive faster than that wait in a queue, and the waiting time grows quickly as
the queue builds up.

That's why the median latency (the typical wait) and the maximum (the worst wait) both matter, and why the rate limits
you give each client should add up to no more than the service can actually handle. The next cell measures latency on the
real server; the last exercise turns it into a capacity plan.

In [ ]:
latencies_ms = []
for question in ["How much is the funeral grant?", "What is the retirement age?", "Who pays for work injuries?",
                 "How long is maternity benefit paid for?", "When are March contributions due?"]:
    start = time.perf_counter()
    requests.post(BASE + "/v1/ask", json={"question": question}, headers={"X-API-Key": "lab-key-2026"}, timeout=120)
    latencies_ms.append((time.perf_counter() - start) * 1000)      # seconds -> milliseconds

latencies = pd.Series(latencies_ms)
print(f"latency over {len(latencies)} questions: median {latencies.median():.0f} ms, max {latencies.max():.0f} ms")

Almost all of that time is the LLM generating the answer. Retrieval takes milliseconds. That tells you where to spend money
if it's too slow: a GPU (graphics processing unit) for the LLM server, or a smaller model. On Day 10 you'll put these
latencies on a monitoring dashboard.

Always stop a background server when you're done with it, or it keeps running (and holding port 8010) until you restart
the computer. `terminate()` asks it to stop; `wait()` waits until it has.

In [ ]:
# Stop the background server.
server.terminate()
server.wait(timeout=20)
print("server stopped")

> **Where we are.** The desk application calls the service, retries correctly after `429`, and you've measured the
> latency. You also found that Swahili questions can be refused because the threshold was set on English questions only.
> The server is stopped. Last, list what's simplified here.

## 6. Before this goes live

What you built is the right *shape* for a production service, but a few things are deliberately simplified. Before real use:

| Simplified here | In production |
|---|---|
| API keys in an environment variable | Keys in a secrets manager (e.g. HashiCorp Vault, Azure Key Vault), rotated regularly; or the Fund's single sign-on (OAuth2/OIDC) for staff-facing apps |
| Plain HTTP on the laptop | HTTPS only, behind the Fund's API gateway or reverse proxy |
| Rate limit in memory, one process | Rate limiting in the API gateway or a shared store (e.g. Redis), so it works across several servers |
| Log file on disk | Logs shipped to the central logging system, with a retention period agreed with the data protection officer |
| One process | Several replicas behind a load balancer, health-checked (Day 10 covers containers and Kubernetes) |

**Your turn (3).** The call-centre system will send about 2,000 questions a day, most between 08:00 and 16:00, with peaks
of about 40 a minute. Using the latency you measured, how many requests can one server handle per minute if the LLM
answers one at a time? What rate limit would you give the call centre, and what would you need to change to handle their
peak?

*Your answer here.*

---

## Looking back, looking ahead

**What you did in this lab.** You turned the handbook assistant into an internal API: the Lab 1 code as a class loaded once,
a FastAPI app with a versioned contract, API keys, per-client rate limits and privacy-aware logging, tests of every
safeguard with `TestClient`, a real server run with uvicorn, a desk application that respects `Retry-After`, a latency
measurement, and a list of what changes before production.

**The ideas to carry forward.**

- A service hides how it works behind a small, versioned contract.
- Authenticate, rate-limit and log every internal API; don't log personal data.
- Test the safeguards automatically, without a server.
- Most of the latency is the LLM; that's where speed costs money.
- Report quality by language: a threshold set in English can quietly fail Swahili speakers.

**What happens next.** Lab 3: what would it cost, and what would change, if the same pipeline ran on a cloud platform
instead of the Fund's own servers?

## Suggested answer to (1)

A new endpoint can be added to the app that's already loaded in this notebook. The address `/v1/sources/{source}` has a
**path parameter**: whatever is written in place of `{source}` arrives as the function's `source` argument.
`Depends(main.require_key)` protects it with the same key check as `/v1/ask`. An unknown source gets `404 Not Found`.

In [ ]:
from fastapi import Depends, HTTPException


# Add a new endpoint to the running app. Depends(main.require_key) protects it with the same key check.
@main.app.get("/v1/sources/{source}")
def list_sections(source: str, client: str = Depends(main.require_key)):
    sections = []
    for chunk in main.assistant.chunks:
        if chunk["source"] == source:
            sections.append(chunk["section"])
    if len(sections) == 0:
        raise HTTPException(status_code=404, detail="Unknown source")
    return {"source": source, "sections": sections}


CC_KEY = {"X-API-Key": "cc-test-key-456"}
print(client.get("/v1/sources/07_funeral_grant.md", headers=CC_KEY).json())
print(client.get("/v1/sources/nothing.md", headers=CC_KEY).status_code)

A desk officer who sees a citation like `04_old_age_pension.md 4.3` wants to know what else the assistant can draw on. A
sources endpoint also lets the document owners check that a new circular really made it into the index.

## Suggested answer to (2)

The assistant object is still loaded in this notebook (from the `TestClient` section), so you can score questions directly.
`search(q, k=1)` returns a list with one `(chunk, similarity)` pair; `[0][1]` takes that pair's similarity.

In [ ]:
pairs = [
    ("How long is maternity benefit paid for?", "Fao la uzazi linalipwa kwa muda gani?"),
    ("How much is the funeral grant?", "Msaada wa mazishi ni kiasi gani?"),
    ("What is the deadline to claim unemployment benefit?", "Mwisho wa kudai fao la kukosa ajira ni lini?"),
    ("What is the minimum voluntary contribution?", "Mchango wa chini kwa mwanachama wa hiari ni kiasi gani?"),
    ("Can pensioners verify using the mobile app?", "Je, wastaafu wanaweza kufanya uhakiki kwa programu ya simu?"),
]

rows = []
for english, swahili in pairs:
    english_score = main.assistant.search(english, k=1)[0][1]      # [0] = best result, [1] = its similarity
    swahili_score = main.assistant.search(swahili, k=1)[0][1]
    rows.append({"English": english, "score EN": english_score, "Swahili": swahili, "score SW": swahili_score})

print("current threshold:", main.assistant.cfg["min_score"])
print("off-topic question score:", round(main.assistant.search("What is the capital of Kenya?", k=1)[0][1], 3))
pd.DataFrame(rows).round(3)

If the Swahili scores sit between the off-topic score and the current threshold, a lower threshold for Swahili questions
(or for all questions) lets them through. Check the unanswerable English questions again after any change, because a lower
bar lets more of those reach the LLM too, where the prompt's refusal rule is the only thing left to stop them.